from utils import setup_korean_font
setup_korean_font()
# 05 CNN + Stacking + 운용점 분석 — Phase 5

**CNN:** 25 피처를 1D 시퀀스로 처리 (PyTorch Conv1d×2 → FC, Dropout=0.3)  
**Stacking:** LR-L2/QDA/RF/GBM/MLP OOF 예측 → 메타 LR-L2 (data leakage 방지)  
**운용점:** Precision=0.95/0.99 고정 시 Recall + PR 곡선 마커

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns

from utils import set_seed, setup_korean_font
set_seed(42)
setup_korean_font()

FIGURES_DIR = PROJECT_ROOT / 'results' / 'figures'
TABLES_DIR  = PROJECT_ROOT / 'results' / 'tables'
sns.set_theme(style='whitegrid', palette='muted')
setup_korean_font()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Setup OK')

Setup OK


## Phase 5 결과 로드

In [2]:
p5 = pd.read_csv(TABLES_DIR / 'phase5_results.csv')
ops = pd.read_csv(TABLES_DIR / 'operating_points.csv')

# Phase 4 reference
phase4_ref = pd.DataFrame([
    {'model':'MLP[256,128,64] (Ph4)','roc_auc_mean':0.9497,'roc_auc_std':0.0166,'pr_auc_mean':0.4710,'pr_auc_std':0.0968},
    {'model':'RF (Ph4)',            'roc_auc_mean':0.9478,'roc_auc_std':0.0123,'pr_auc_mean':0.4481,'pr_auc_std':0.1121},
])

combined = pd.concat([p5, phase4_ref], ignore_index=True)
print('=== Phase 5 결과 (Phase 4 참조 포함) ===')
print(combined[['model','roc_auc_mean','roc_auc_std','pr_auc_mean','pr_auc_std']]
      .sort_values('roc_auc_mean', ascending=False).to_string(index=False))

stacking_beat = p5[p5['model'].str.contains('Stacking')].iloc[0]['roc_auc_mean'] > 0.9497
print(f'\nStacking > single best(MLP 0.9497)? {"YES" if stacking_beat else "NO"}')

=== Phase 5 결과 (Phase 4 참조 포함) ===
                model  roc_auc_mean  roc_auc_std  pr_auc_mean  pr_auc_std
MLP[256,128,64] (Ph4)        0.9497       0.0166       0.4710      0.0968
             RF (Ph4)        0.9478       0.0123       0.4481      0.1121
         CNN1D(32,64)        0.9472       0.0152       0.4501      0.1194
    Stacking(LR-meta)        0.9462       0.0108       0.4484      0.1021
         CNN1D(16,32)        0.9408       0.0133       0.4126      0.1139

Stacking > single best(MLP 0.9497)? NO


## Phase 5 요약 그림

In [3]:
img = mpimg.imread(str(FIGURES_DIR / 'phase5_summary.png'))
fig, ax = plt.subplots(figsize=(15, 6))
ax.imshow(img); ax.axis('off')
plt.tight_layout()
plt.show()

PR-AUC 기준으로 Stacking(0.4484)은 RF(0.4481)와 거의 동일하고 MLP(0.4710)에는 미치지 못했다 — base learner 간 다양성이 충분하지 않으면 스태킹의 앙상블 효과가 제한된다. 운용점 분석에서는 Precision≥0.99 조건 하에 MLP Recall=32.4%, Stacking Recall=23.9%로, 불량 제로를 목표로 하는 산업 현장 운용 시 MLP가 더 유리하다.

## 운용점 상세표

In [4]:
print('=== 운용점 분석 (Precision 고정 -> Recall)')
print(ops.to_string(index=False))
print()
print('[해석]')
print('  Precision>=0.95: 불량 예측 100건 중 95건 이상 실제 불량')
print('  Precision>=0.99: 불량 예측 100건 중 99건 이상 실제 불량')
print()
mlp_rec99 = ops[(ops['model']=='MLP[256,128,64]') & (ops['target_precision']=='prec>=0.99')]['recall_at_target'].values[0]
stack_rec99 = ops[(ops['model']=='Stacking') & (ops['target_precision']=='prec>=0.99')]['recall_at_target'].values[0]
print(f'  Precision>=0.99 시: MLP Recall={mlp_rec99:.3f}  Stacking Recall={stack_rec99:.3f}')
print(f'  -> 불량 {int(71*mlp_rec99)}/{71}건 탐지 가능 (MLP, 전체 71건 중)')

=== 운용점 분석 (Precision 고정 -> Recall)
          model target_precision  achieved_precision  recall_at_target  threshold
       Stacking       prec>=0.95              0.9583            0.3239     0.5443
       Stacking       prec>=0.99              1.0000            0.2394     0.5825
MLP[256,128,64]       prec>=0.95              0.9600            0.3380     0.9977
MLP[256,128,64]       prec>=0.99              1.0000            0.3239     0.9990

[해석]
  Precision>=0.95: 불량 예측 100건 중 95건 이상 실제 불량
  Precision>=0.99: 불량 예측 100건 중 99건 이상 실제 불량

  Precision>=0.99 시: MLP Recall=0.324  Stacking Recall=0.239
  -> 불량 22/71건 탐지 가능 (MLP, 전체 71건 중)


## 전 Phase 누적 비교

In [5]:
all_phases = pd.DataFrame([
    {'Phase':'Ph2 (전처리 기준)','model':'LR-L2+SMOTE','roc_auc_mean':0.9311,'pr_auc_mean':0.2413},
    {'Phase':'Ph3','model':'QDA(reg=0.01)','roc_auc_mean':0.9344,'pr_auc_mean':0.3526},
    {'Phase':'Ph4-A','model':'TreeTop-15+LR','roc_auc_mean':0.9380,'pr_auc_mean':0.2899},
    {'Phase':'Ph4-B','model':'MLP[256,128,64]','roc_auc_mean':0.9497,'pr_auc_mean':0.4710},
    {'Phase':'Ph5','model':'CNN1D(32,64)','roc_auc_mean':0.9472,'pr_auc_mean':0.4501},
    {'Phase':'Ph5','model':'Stacking','roc_auc_mean':0.9462,'pr_auc_mean':0.4484},
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
COLORS = ['#8c8c8c','#2ca02c','#ff7f0e','#1f77b4','#d62728','#9467bd']

for ax, metric, ylabel in [(axes[0],'roc_auc_mean','ROC-AUC'),(axes[1],'pr_auc_mean','PR-AUC')]:
    bars = ax.barh(all_phases['model'], all_phases[metric], color=COLORS, alpha=0.85)
    for bar, val in zip(bars, all_phases[metric]):
        ax.text(val+0.001, bar.get_y()+bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=8.5)
    ax.set_xlabel(ylabel, fontsize=11)
    ax.set_title(f'전 Phase 누적 {ylabel} 비교', fontsize=12)
    ax.set_xlim([0.8, 1.0] if metric=='roc_auc_mean' else [0, 0.6])

plt.suptitle('Phase 2-5 단계별 Ablation 성능 누적 비교', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'all_phases_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: all_phases_comparison.png')

Saved: all_phases_comparison.png


## 결론

In [6]:
print('=' * 65)
print('Phase 5 결론 + 전체 Ablation 요약')
print('=' * 65)
print("""
[Phase 5 발견]
1. CNN1D(32,64): ROC=0.9472, PR=0.4501 - Phase 4 앙상블 수준 경쟁력
2. Stacking: ROC=0.9462, PR=0.4484 - 단일 MLP(0.9497) 미초과
   -> base learner 다양성 부족. SVM-RBF 추가 시 개선 여지.
3. 운용점(Precision>=0.99): MLP Recall=32.4% > Stacking 23.9%
   -> 최고 정밀도 요구 시 MLP가 실용적 우위.

[전 Phase 흐름 요약]
  Ph2 전처리: +0.024 ROC-AUC (SMOTE 효과)
  Ph3 선형:   QDA로 PR-AUC 0.3526 달성 (가이드북 SVM 비판)
  Ph4 앙상블: MLP로 PR-AUC 0.4710 (+0.12)
  Ph5 CNN/Stack: 단일 MLP를 넘지 못함 -> '단일 best는 MLP'

[가이드북 vs 우리]
  가이드북: AE/SVM/DNN 순서로 실험 -> 단순 재현
  우리: 각 단계(전처리/선형/DR/앙상블/CNN/Stack)를 분리해 어느
       단계가 성능을 좌우했는지 정량화 -> originality 확보
""")

Phase 5 결론 + 전체 Ablation 요약

[Phase 5 발견]
1. CNN1D(32,64): ROC=0.9472, PR=0.4501 - Phase 4 앙상블 수준 경쟁력
2. Stacking: ROC=0.9462, PR=0.4484 - 단일 MLP(0.9497) 미초과
   -> base learner 다양성 부족. SVM-RBF 추가 시 개선 여지.
3. 운용점(Precision>=0.99): MLP Recall=32.4% > Stacking 23.9%
   -> 최고 정밀도 요구 시 MLP가 실용적 우위.

[전 Phase 흐름 요약]
  Ph2 전처리: +0.024 ROC-AUC (SMOTE 효과)
  Ph3 선형:   QDA로 PR-AUC 0.3526 달성 (가이드북 SVM 비판)
  Ph4 앙상블: MLP로 PR-AUC 0.4710 (+0.12)
  Ph5 CNN/Stack: 단일 MLP를 넘지 못함 -> '단일 best는 MLP'

[가이드북 vs 우리]
  가이드북: AE/SVM/DNN 순서로 실험 -> 단순 재현
  우리: 각 단계(전처리/선형/DR/앙상블/CNN/Stack)를 분리해 어느
       단계가 성능을 좌우했는지 정량화 -> originality 확보



전 Phase를 통틀어 MLP[256,128,64]+SMOTE가 ROC-AUC 0.9497·PR-AUC 0.4710으로 최고 성능을 기록했다. Dropout 추가가 Guidebook DNN(0.9468) 재현치를 상회하는 핵심 요인이며, 불균형 처리(Ph2)와 비선형 모델 도입(Ph4)이 성능 향상을 이끈 두 단계임을 단계별 수치 비교로 확인했다.